- 目标：当前文件夹下宽度大于 1000 的图片
- 调整方法：维持长宽比，宽度调整为 900，尽量维持图片清晰度
- 输出：
  - 原始图片：存入 '.\fig-raw' 文件夹，作为备份
  - 转换后的图片：覆盖当前路径下同名文件，删除旧文件
- 要求：确保代码可以独立运行，不依赖于特定的库或环境配置

In [2]:
from pathlib import Path
from PIL import Image, ImageOps

# 兼容：若下方单元尚未执行，则补充导入
if "Path" not in globals():
if "Image" not in globals() or "ImageOps" not in globals():

# 参数（优先复用已有变量）
folder = globals().get("folder", Path.cwd())
SUPPORTED_EXTS = globals().get("SUPPORTED_EXTS", {".jpg", ".jpeg", ".png", ".webp", ".tif", ".tiff"})
MIN_WIDTH = globals().get("MIN_WIDTH", 1000)
TARGET_WIDTH = globals().get("TARGET_WIDTH", 900)

# 兼容不同 Pillow 版本
RESAMPLE = Image.Resampling.LANCZOS if hasattr(Image, "Resampling") else Image.LANCZOS

# 备份目录
backup_dir = folder / "fig-raw"
backup_dir.mkdir(exist_ok=True)

processed = 0
skipped = 0
errors = 0

for img_path in folder.iterdir():
    if not img_path.is_file() or img_path.suffix.lower() not in SUPPORTED_EXTS:
        continue

    try:
        # 先备份原图（同名覆盖）
        backup_path = backup_dir / img_path.name
        backup_path.write_bytes(img_path.read_bytes())

        with Image.open(img_path) as im:
            im = ImageOps.exif_transpose(im)
            w, h = im.size

            if w <= MIN_WIDTH:
                skipped += 1
                continue

            new_h = int(h * (TARGET_WIDTH / w))
            resized = im.resize((TARGET_WIDTH, new_h), RESAMPLE)

            ext = img_path.suffix.lower()
            save_kwargs = {}

            if ext in {".jpg", ".jpeg"}:
                if resized.mode not in ("RGB", "L"):
                    resized = resized.convert("RGB")
                save_kwargs = {"quality": 95, "subsampling": 0, "optimize": True}
            elif ext == ".png":
                save_kwargs = {"optimize": True, "compress_level": 6}
            elif ext == ".webp":
                save_kwargs = {"quality": 95, "method": 6}
            elif ext in {".tif", ".tiff"}:
                save_kwargs = {"compression": "tiff_lzw"}

            # 临时文件写入后替换原文件（覆盖同名并删除旧文件）
            temp_path = img_path.with_name(f"{img_path.stem}.__tmp__{img_path.suffix}")
            resized.save(temp_path, **save_kwargs)
            temp_path.replace(img_path)

            processed += 1
            print(f"已处理: {img_path.name}  {w}x{h} -> {TARGET_WIDTH}x{new_h}")

    except Exception as e:
        errors += 1
        print(f"处理失败: {img_path.name}，原因: {e}")

print(f"\n完成：处理 {processed} 张，跳过 {skipped} 张，失败 {errors} 张。")
print(f"原图备份目录: {backup_dir}")

IndentationError: expected an indented block after 'if' statement on line 5 (743219574.py, line 6)

In [5]:
from pathlib import Path
from shutil import copy2
from PIL import Image, ImageOps

# 可独立运行：仅依赖 Python 标准库 + Pillow


# 参数
folder = Path.cwd()  # 当前文件夹
SUPPORTED_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".tif", ".tiff"}
MIN_WIDTH = 1000
TARGET_WIDTH = 900

# 兼容不同 Pillow 版本
RESAMPLE = Image.Resampling.LANCZOS if hasattr(Image, "Resampling") else Image.LANCZOS

# 备份目录
backup_dir = folder / "fig-raw"
backup_dir.mkdir(exist_ok=True)

processed = 0
skipped = 0
errors = 0

for img_path in folder.iterdir():
    if not img_path.is_file() or img_path.suffix.lower() not in SUPPORTED_EXTS:
        continue

    try:
        # 读取图片尺寸（先判断是否需要处理）
        with Image.open(img_path) as im:
            im = ImageOps.exif_transpose(im)
            w, h = im.size

            if w <= MIN_WIDTH:
                skipped += 1
                continue

            # 需要处理时先备份原图（同名覆盖）
            backup_path = backup_dir / img_path.name
            copy2(img_path, backup_path)

            # 按比例缩放到目标宽度
            new_h = int(h * (TARGET_WIDTH / w))
            resized = im.resize((TARGET_WIDTH, new_h), RESAMPLE)

            ext = img_path.suffix.lower()
            save_kwargs = {}

            if ext in {".jpg", ".jpeg"}:
                if resized.mode not in ("RGB", "L"):
                    resized = resized.convert("RGB")
                save_kwargs = {"quality": 95, "subsampling": 0, "optimize": True}
            elif ext == ".png":
                save_kwargs = {"optimize": True, "compress_level": 6}
            elif ext == ".webp":
                save_kwargs = {"quality": 95, "method": 6}
            elif ext in {".tif", ".tiff"}:
                save_kwargs = {"compression": "tiff_lzw"}

            # 临时文件写入后替换原文件（覆盖同名并删除旧文件）
            temp_path = img_path.with_name(f"{img_path.stem}.__tmp__{img_path.suffix}")
            resized.save(temp_path, **save_kwargs)
            temp_path.replace(img_path)

            processed += 1
            print(f"已处理: {img_path.name}  {w}x{h} -> {TARGET_WIDTH}x{new_h}")

    except Exception as e:
        errors += 1
        print(f"处理失败: {img_path.name}，原因: {e}")

print(f"\n完成：处理 {processed} 张，跳过 {skipped} 张，失败 {errors} 张。")
print(f"原图备份目录: {backup_dir}")

已处理: ci-policy-Lane-2025-intro-outline.png  1491x1055 -> 900x636
已处理: ci-policy-Lian-2026-Government-fund-topic-selection.png  1491x1055 -> 900x636

完成：处理 2 张，跳过 0 张，失败 0 张。
原图备份目录: d:\github_lianxh\ci-policy\figs\fig-raw


In [1]:
from pathlib import Path
from PIL import Image, ImageOps



# 参数配置
folder = Path.cwd()  # 当前文件夹
# folder = Path(r"D:\JG\助教推文提交\2020助教推文\已完成\连小白-丁闪闪\ChatGPT、Codex、Copilot、Claude Code：AI 工具如何分工协作？\ai_agent_tool_images")  # 替换为目标文件夹路径
SUPPORTED_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".tif", ".tiff"}
MIN_WIDTH = 1000
TARGET_WIDTH = 900

# 兼容不同 Pillow 版本
RESAMPLE = Image.Resampling.LANCZOS if hasattr(Image, "Resampling") else Image.LANCZOS

processed = 0
skipped = 0
errors = 0

for img_path in folder.iterdir():
    if not img_path.is_file() or img_path.suffix.lower() not in SUPPORTED_EXTS:
        continue

    try:
        with Image.open(img_path) as im:
            im = ImageOps.exif_transpose(im)
            w, h = im.size

            if w <= MIN_WIDTH:
                skipped += 1
                continue

            new_h = int(h * (TARGET_WIDTH / w))
            resized = im.resize((TARGET_WIDTH, new_h), RESAMPLE)

            ext = img_path.suffix.lower()
            save_kwargs = {}

            if ext in {".jpg", ".jpeg"}:
                if resized.mode not in ("RGB", "L"):
                    resized = resized.convert("RGB")
                save_kwargs = {"quality": 95, "subsampling": 0, "optimize": True}
            elif ext == ".png":
                save_kwargs = {"optimize": True, "compress_level": 6}
            elif ext == ".webp":
                save_kwargs = {"quality": 95, "method": 6}
            elif ext in {".tif", ".tiff"}:
                save_kwargs = {"compression": "tiff_lzw"}

            # 临时文件写入后替换原文件，确保“覆盖同名文件，删除旧文件”
            temp_path = img_path.with_name(f"{img_path.stem}.__tmp__{img_path.suffix}")
            resized.save(temp_path, **save_kwargs)
            temp_path.replace(img_path)

            processed += 1
            print(f"已处理: {img_path.name}  {w}x{h} -> {TARGET_WIDTH}x{new_h}")

    except Exception as e:
        errors += 1
        print(f"处理失败: {img_path.name}，原因: {e}")

print(f"\n完成：处理 {processed} 张，跳过 {skipped} 张，失败 {errors} 张。")

已处理: lianxh_codex_agents_md_workflow_handdrawn.png  1536x1024 -> 900x600
已处理: lianxh_codex_plugin_ecosystem_handdrawn.png  1536x1024 -> 900x600
已处理: lianxh_codex_project_folder_structure_handdrawn.png  1536x1024 -> 900x600

完成：处理 3 张，跳过 0 张，失败 0 张。
